# 편의시설 점수 기준 수립을 위한 탐색적 데이터 분석 (EDA)

이 노트는 매물과 각종 편의시설(대형마트, 편의점, 세탁소, 공원 등) 간의 거리 분포를 분석하여, '편의/생활 온도' 점수 산정의 최적 기준(만점 거리, 0점 거리 등)과 가중치를 결정하기 위해 작성되었습니다.

In [ ]:
import os
import json
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial import cKDTree
from typing import List, Dict, Tuple

# 한글 폰트 설정 (Windows 기준)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

## 1. 데이터 로드

분석에는 다음 데이터가 필요합니다:
1. **매물 샘플**: 점수 시뮬레이션을 위한 서울시 원룸/투룸 매물 샘플 (약 5,000개)
2. **편의시설 데이터**: 
   - 대형마트 (백화점, 쇼핑센터 포함)
   - 편의점, 세탁소 (소상공인 데이터)
   - 공원
   - 문화시설

In [ ]:
# 경로 설정
PROJECT_ROOT = os.path.abspath("../../../") # analytics/temperature/convenience 기준
DATA_DIR = os.path.join(PROJECT_ROOT, "data", "GraphDB_data")
RAW_DATA_DIR = os.path.join(PROJECT_ROOT, "scripts", "dataCrawling", "피터팬 매물 데이터", "data")

print(f"프로젝트 루트: {PROJECT_ROOT}")
print(f"데이터 경로: {DATA_DIR}")

def load_properties(limit=5000):
    """매물 데이터 로드 (위도/경도 포함한 경량 파일 사용)"""
    geo_file = os.path.join(DATA_DIR, "land", "00_통합_원투룸.json")
    print(f"매물 로드 중: {geo_file}")
    
    # 파일이 없으면 빈 DataFrame 반환
    if not os.path.exists(geo_file):
        print("파일을 찾을 수 없습니다.")
        return pd.DataFrame()
    
    with open(geo_file, 'r', encoding='utf-8') as f:
        geo_data = json.load(f)
        
    df = pd.DataFrame(geo_data)
    
    # 좌표 정보 추출
    df['lat'] = df['좌표_정보'].apply(lambda x: x.get('위도') if x else None)
    df['lon'] = df['좌표_정보'].apply(lambda x: x.get('경도') if x else None)
    df = df.dropna(subset=['lat', 'lon'])
    
    # 샘플링
    if limit:
        df = df.sample(n=min(limit, len(df)), random_state=42)
        
    print(f"총 {len(df)}개 매물 로드 완료.")
    return df[['매물번호', 'lat', 'lon']]

properties_df = load_properties(limit=5000)
properties_df.head()

In [ ]:
def load_facilities():
    """각종 편의시설 데이터 로드"""
    facilities = {}
    
    # 1. 대형마트 (Mart)
    mart_path = os.path.join(DATA_DIR, "store_data", "서울시 대규모점포 인허가 정보.csv")
    try:
        df = pd.read_csv(mart_path, encoding='cp949')
    except:
        df = pd.read_csv(mart_path, encoding='euc-kr')
    
    cats = ['대형마트', '백화점', '쇼핑센터', '복합쇼핑몰', '구분없음']
    df = df[df['업태구분명'].isin(cats)]
    df = df.dropna(subset=['위도', '경도'])
    facilities['mart'] = df[['사업장명', '위도', '경도']].rename(columns={'위도': 'lat', '경도': 'lon', '사업장명': 'name'})
    
    # 2. 소상공인 상가 (편의점, 세탁소)
    store_path = os.path.join(DATA_DIR, "store_data", "소상공인시장진흥공단_상가(상권)정보_서울_cleaned.csv")
    try:
        df_store = pd.read_csv(store_path, encoding='utf-8')
    except:
        df_store = pd.read_csv(store_path, encoding='cp949')
        
    # 세탁소
    laundry = df_store[df_store['상권업종소분류명'] == '세탁소'].dropna(subset=['위도', '경도'])
    facilities['laundry'] = laundry[['상호명', '위도', '경도']].rename(columns={'위도': 'lat', '경도': 'lon', '상호명': 'name'})
    
    # 편의점
    conv = df_store[df_store['상권업종소분류명'] == '편의점'].dropna(subset=['위도', '경도'])
    facilities['convenience'] = conv[['상호명', '위도', '경도']].rename(columns={'위도': 'lat', '경도': 'lon', '상호명': 'name'})
    
    # 3. 공원
    park_files = glob.glob(os.path.join(DATA_DIR, "park", "*.csv"))
    park_dfs = []
    for f in park_files:
        try:
            d = pd.read_csv(f, encoding='cp949')
        except:
            d = pd.read_csv(f, encoding='utf-8')
        park_dfs.append(d)
    
    if park_dfs:
        df_park = pd.concat(park_dfs, ignore_index=True)
        df_park = df_park.dropna(subset=['위도', '경도'])
        facilities['park'] = df_park[['공원명', '위도', '경도']].rename(columns={'위도': 'lat', '경도': 'lon', '공원명': 'name'})
        
    # 4. 문화시설
    culture_path = os.path.join(DATA_DIR, "culture", "서울시 문화공간 정보.csv")
    try:
        df_cul = pd.read_csv(culture_path, encoding='utf-8')
    except:
        df_cul = pd.read_csv(culture_path, encoding='euc-kr')
    df_cul = df_cul.dropna(subset=['위도', '경도'])
    facilities['culture'] = df_cul[['문화시설명', '위도', '경도']].rename(columns={'위도': 'lat', '경도': 'lon', '문화시설명': 'name'})
    
    return facilities

facilities = load_facilities()
for k, v in facilities.items():
    print(f"{k}: {len(v)} 개 데이터 로드됨")

## 2. 거리 분석 (Distance Analysis)

각 매물에서 가장 가까운 시설까지의 직선 거리를 계산합니다.

In [ ]:
def get_nearest_distance(sources_df, targets_df):
    if targets_df.empty:
        return np.full(len(sources_df), np.inf)
    
    # KDTree를 이용한 빠른 최단 거리 검색
    # 위경도 -> 미터 변환 (서울 근사치: 경도 1도=88km, 위도 1도=111km)
    
    def to_xy(df):
        x = df['lon'].values * 88000
        y = df['lat'].values * 111000
        return np.column_stack([x, y])
    
    source_xy = to_xy(sources_df)
    target_xy = to_xy(targets_df)
    
    tree = cKDTree(target_xy)
    dists, _ = tree.query(source_xy, k=1)
    return dists

results = properties_df.copy()

for fac_name, fac_df in facilities.items():
    print(f"{fac_name}까지의 거리 계산 중...")
    results[f'dist_{fac_name}'] = get_nearest_distance(results, fac_df)
    
results.head()

## 3. 시각화 및 통계 (Visualization & Percentiles)

누적 분포(CDF)와 히스토그램을 통해 거리 분포를 확인합니다.
이를 통해 '대부분의 매물이 5분(300m) 거리에 있다' 등을 파악할 수 있습니다.

In [ ]:
def analyze_Metric(df, col_name, title, limit=2000):
    data = df[col_name]
    # 시각화를 위해 이상치(너무 먼 거리) 필터링
    data_plot = data[data <= limit]
    
    plt.figure(figsize=(12, 5))
    
    # 히스토그램
    plt.subplot(1, 2, 1)
    sns.histplot(data_plot, bins=50, kde=True)
    plt.title(f"{title} 거리 분포 (<= {limit}m)")
    plt.xlabel("거리 (m)")
    plt.ylabel("매물 수")
    
    # CDF (누적 분포)
    plt.subplot(1, 2, 2)
    sns.ecdfplot(data=data_plot)
    plt.title(f"{title} 누적 분포 (CDF)")
    plt.xlabel("거리 (m)")
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()
    
    # 통계 수치 출력
    print(f"--- {title} 거리 통계 ---")
    print(data.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95]))
    print(f"200m 이내 비율: {(data <= 200).mean()*100:.1f}%")
    print(f"500m 이내 비율: {(data <= 500).mean()*100:.1f}%")
    print(f"1000m 이내 비율: {(data <= 1000).mean()*100:.1f}%")
    print("\n")

# 각 항목별 분석 실행
analyze_Metric(results, 'dist_convenience', '편의점', limit=1000)
analyze_Metric(results, 'dist_laundry', '세탁소', limit=1000)
analyze_Metric(results, 'dist_mart', '대형마트', limit=3000)
analyze_Metric(results, 'dist_park', '공원', limit=2000)
analyze_Metric(results, 'dist_culture', '문화시설', limit=3000)

## 4. 점수 기준 제안 (Proposal)

위의 분포를 바탕으로 점수 산정 공식을 제안할 수 있습니다.
예를 들어 편의점이 90% 이상 500m 이내에 있다면, 500m를 0점 기준으로 삼을 수 있습니다.

### 예시 로직
- **편의점**: 200m 이내 만점, 500m 이상 0점
- **대형마트**: 500m 이내 만점, 1.5km 이상 0점

수식: `score = max(0, 1 - (distance / threshold)) * max_points`